# FrostAnalitic — Análisis de Ciencia de Datos
**Proyecto Final — Ciencia de Datos**

Este notebook documenta el proceso completo de análisis de datos aplicado al sistema experto de diagnóstico de equipos de refrigeración **FrostAnalitic**.

---
### Objetivo
Demostrar que un sistema de diagnóstico basado en reglas (árbol de decisiones hardcodeado) puede ser mejorado mediante técnicas de Machine Learning entrenadas con los datos que el propio sistema acumula durante su uso.

### Pregunta de investigación
> ¿Puede un sistema experto de diagnóstico de refrigeración mejorar su precisión mediante el aprendizaje automático a partir de la retroalimentación de sus usuarios?


## 1. Importación de librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
import pickle
import os

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    accuracy_score, f1_score,
    classification_report, confusion_matrix
)

warnings.filterwarnings('ignore')

# Estilo de gráficas
plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor']   = '#161820'
plt.rcParams['axes.edgecolor']   = '#2a2d3a'
plt.rcParams['text.color']       = '#e4e6f0'
plt.rcParams['axes.labelcolor']  = '#9ca3af'
plt.rcParams['xtick.color']      = '#9ca3af'
plt.rcParams['ytick.color']      = '#9ca3af'
plt.rcParams['grid.color']       = '#1c1e28'
plt.rcParams['font.family']      = 'sans-serif'

print('✓ Librerías cargadas correctamente')
print(f'  pandas     {pd.__version__}')
print(f'  numpy      {np.__version__}')
print(f'  sklearn    importado')

## 2. Carga y exploración del dataset

El dataset fue generado por `simulate_data.py`, que simula 600 diagnósticos reales basados en el comportamiento del árbol de decisiones de FrostAnalitic.

Cada registro representa un diagnóstico completo con:
- El **equipo** diagnosticado
- El **síntoma** detectado en el árbol
- La **falla diagnosticada** por el sistema
- La **falla real** (ground truth)
- Si el diagnóstico **fue correcto** o no

In [ ]:
# Cargar dataset — ajusta la ruta si es necesario
BASE = os.path.dirname(os.path.abspath('__file__'))
CSV  = os.path.join(BASE, 'dataset_diagnosticos.csv')

df = pd.read_csv(CSV)

print(f'Registros totales: {len(df)}')
print(f'Columnas: {list(df.columns)}')
print(f'\nPrimeras 5 filas:')
df.head()

In [ ]:
# Información general del dataset
print('=== INFORMACIÓN DEL DATASET ===')
print(f'Total de diagnósticos:     {len(df)}')
print(f'Tipos de equipo:           {df["equipo"].nunique()}')
print(f'Tipos de síntoma:          {df["sintoma"].nunique()}')
print(f'Fallas distintas:          {df["falla_correcta_id"].nunique()}')
print(f'Período simulado:          {df["fecha"].min()[:10]} → {df["fecha"].max()[:10]}')
print(f'\nDiagnósticos correctos:    {df["fue_correcto"].sum()} ({df["fue_correcto"].mean()*100:.1f}%)')
print(f'Diagnósticos incorrectos:  {(~df["fue_correcto"].astype(bool)).sum()} ({(1-df["fue_correcto"].mean())*100:.1f}%)')

## 3. Análisis exploratorio de datos (EDA)

In [ ]:
# Distribución por equipo
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Distribución del Dataset', color='#e4e6f0', fontsize=13, fontweight='bold')

# Gráfica 1: por equipo
eq_counts = df['equipo'].value_counts()
colores = ['#4f7fff','#2dd4bf','#fbbf24','#f87171','#a78bfa']
axes[0].barh(eq_counts.index, eq_counts.values, color=colores, height=0.6)
axes[0].set_title('Diagnósticos por equipo', color='#e4e6f0')
axes[0].set_xlabel('Cantidad')
for i, v in enumerate(eq_counts.values):
    axes[0].text(v+1, i, str(v), va='center', color='#e4e6f0', fontsize=10)

# Gráfica 2: correctos vs incorrectos
vals = [df['fue_correcto'].sum(), (~df['fue_correcto'].astype(bool)).sum()]
labels = [f'Correctos\n{vals[0]}', f'Incorrectos\n{vals[1]}']
axes[1].pie(vals, labels=labels, colors=['#34d399','#f87171'],
            autopct='%1.1f%%', startangle=90,
            textprops={'color':'#e4e6f0'})
axes[1].set_title('Precisión del árbol de reglas', color='#e4e6f0')

plt.tight_layout()
plt.show()

In [ ]:
# Top 10 fallas más frecuentes
FALLA_NOMBRES = {
    1:'Fuga refrigerante',    2:'Capilar obstruido',
    3:'Compresor defect.',    4:'Relay/capacitor',
    5:'Termostato',           6:'Vent. evaporador',
    7:'Vent. condensador',    8:'Condensador sucio',
    9:'Drenaje obstruido',   10:'Empaque puerta',
   11:'Deshielo defect.',    12:'Tarjeta control',
   13:'Filtros sucios',      14:'Aislamiento',
   15:'Sensor temperatura',  16:'Resist. anti-vaho',
   17:'Falla eléctrica',     18:'Rodamientos'
}

top10 = df['falla_correcta_id'].value_counts().head(10)
nombres = [FALLA_NOMBRES.get(i, f'F{i}') for i in top10.index]

fig, ax = plt.subplots(figsize=(10, 5))
colores_bar = ['#4f7fff' if i < 3 else '#1c2d5a' for i in range(len(top10))]
bars = ax.barh(nombres[::-1], top10.values[::-1], color=colores_bar[::-1], height=0.6)
ax.set_title('Top 10 Fallas Más Frecuentes en el Dataset', color='#e4e6f0', fontsize=12, fontweight='bold')
ax.set_xlabel('Cantidad de diagnósticos')
for bar, val in zip(bars, top10.values[::-1]):
    ax.text(bar.get_width()+0.5, bar.get_y()+bar.get_height()/2,
            str(val), va='center', color='#e4e6f0', fontsize=9)
plt.tight_layout()
plt.show()

print('\nEstadísticas descriptivas:')
print(df[['probabilidad','fue_correcto']].describe().round(2))

In [ ]:
# Precisión por equipo
prec_eq = df.groupby('equipo')['fue_correcto'].agg(['mean','count']).round(3)
prec_eq.columns = ['Precisión', 'Diagnósticos']
prec_eq['Precisión %'] = (prec_eq['Precisión'] * 100).round(1)
prec_eq = prec_eq.sort_values('Precisión', ascending=False)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(prec_eq.index, prec_eq['Precisión %'],
              color='#4f7fff', width=0.5, alpha=0.85)
ax.set_ylim(60, 100)
ax.set_ylabel('Precisión (%)')
ax.set_title('Precisión del Árbol de Reglas por Equipo', color='#e4e6f0', fontsize=12, fontweight='bold')
ax.axhline(df['fue_correcto'].mean()*100, color='#fbbf24',
           linestyle='--', linewidth=1.2, label='Promedio global')
for bar, val in zip(bars, prec_eq['Precisión %']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
            f'{val}%', ha='center', va='bottom', color='#e4e6f0', fontsize=10)
ax.legend()
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

print(prec_eq)

## 4. Preparación de datos para Machine Learning

Para entrenar los modelos necesitamos convertir las variables categóricas (equipo, síntoma) a valores numéricos usando `LabelEncoder`.

In [ ]:
# Codificación de variables categóricas
le_eq   = LabelEncoder()
le_sint = LabelEncoder()

df['equipo_enc']  = le_eq.fit_transform(df['equipo'])
df['sintoma_enc'] = le_sint.fit_transform(df['sintoma'])

# Features y target
X = df[['equipo_enc', 'sintoma_enc', 'probabilidad']].values
y = df['falla_correcta_id'].values

# División train/test (75% / 25%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print('=== DIVISIÓN DEL DATASET ===')
print(f'Total:      {len(X)} registros')
print(f'Entrenamiento: {len(X_train)} ({len(X_train)/len(X)*100:.0f}%)')
print(f'Prueba:        {len(X_test)}  ({len(X_test)/len(X)*100:.0f}%)')
print(f'\nFeatures usados:')
print(f'  - equipo_enc  (equipo codificado: {df["equipo_enc"].nunique()} valores)')
print(f'  - sintoma_enc (síntoma codificado: {df["sintoma_enc"].nunique()} valores)')
print(f'  - probabilidad (confianza del árbol: {X[:,2].min():.0f}% - {X[:,2].max():.0f}%)')
print(f'\nTarget: falla_correcta_id ({len(set(y))} clases distintas)')

## 5. Entrenamiento y comparación de modelos

Se comparan **3 algoritmos** de clasificación contra el árbol de reglas actual:

| Modelo | Descripción |
|--------|-------------|
| Árbol de Decisión | Árbol de clasificación con profundidad máxima 8 |
| Random Forest | Ensemble de 100 árboles con votación mayoritaria |
| Naive Bayes | Clasificador probabilístico bayesiano |

In [ ]:
prec_actual = df['fue_correcto'].mean() * 100

modelos = {
    'Árbol de Decisión': DecisionTreeClassifier(max_depth=8, random_state=42),
    'Random Forest':     RandomForestClassifier(n_estimators=100, random_state=42),
    'Naive Bayes':       GaussianNB(),
}

resultados = {}
print(f'Precisión del árbol de reglas actual: {prec_actual:.1f}%')
print('─' * 55)
print(f'{"Modelo":<25} {"Accuracy":>10} {"F1-Score":>10} {"CV-5":>10}')
print('─' * 55)

for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    acc = accuracy_score(y_test, y_pred) * 100
    f1  = f1_score(y_test, y_pred, average='weighted', zero_division=0) * 100
    cv  = cross_val_score(modelo, X, y, cv=5, scoring='accuracy').mean() * 100
    resultados[nombre] = {'modelo': modelo, 'acc': acc, 'f1': f1, 'cv': cv, 'pred': y_pred}
    print(f'{nombre:<25} {acc:>9.1f}% {f1:>9.1f}% {cv:>9.1f}%')

print('─' * 55)
mejor_nombre = max(resultados, key=lambda k: resultados[k]['acc'])
mejor = resultados[mejor_nombre]
mejora = mejor['acc'] - prec_actual
print(f'\n✓ Mejor modelo: {mejor_nombre} ({mejor["acc"]:.1f}%)')
print(f'✓ Mejora sobre árbol de reglas: +{mejora:.1f}%')

In [ ]:
# Gráfica comparación de modelos
fig, ax = plt.subplots(figsize=(10, 5))

nombres  = ['Árbol de reglas\n(actual)'] + list(resultados.keys())
accs     = [prec_actual] + [resultados[k]['acc'] for k in resultados]
f1s      = [prec_actual] + [resultados[k]['f1']  for k in resultados]
colores  = ['#fbbf24', '#4f7fff', '#34d399', '#f87171']

x = np.arange(len(nombres))
w = 0.35
b1 = ax.bar(x - w/2, accs, w, color=colores, alpha=0.9, label='Accuracy')
b2 = ax.bar(x + w/2, f1s,  w, color=colores, alpha=0.5, label='F1-Score')

ax.set_ylim(40, 108)
ax.set_ylabel('Porcentaje (%)')
ax.set_title('Comparación de Modelos — Accuracy vs F1-Score',
             color='#e4e6f0', fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(nombres, fontsize=10)
ax.axhline(prec_actual, color='#fbbf24', linewidth=1, linestyle='--', alpha=0.6)
ax.legend(facecolor='#1c1e28', edgecolor='#2a2d3a', labelcolor='#e4e6f0')
ax.grid(axis='y', alpha=0.3)

for bar, val in zip(b1, accs):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f'{val:.1f}%', ha='center', va='bottom', color='#e4e6f0', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Evolución de la precisión con el uso

Una de las hipótesis centrales del proyecto es que **el sistema mejora con el uso**. Aquí visualizamos cómo evoluciona la precisión a medida que se acumulan diagnósticos.

In [ ]:
df_s = df.sort_values('sesion_id').copy()
df_s['prec_movil']  = df_s['fue_correcto'].rolling(30, min_periods=5).mean() * 100
df_s['prec_acum']   = df_s['fue_correcto'].expanding().mean() * 100

fig, ax = plt.subplots(figsize=(11, 4.5))

ax.fill_between(df_s['sesion_id'], df_s['prec_movil'],
                alpha=0.1, color='#4f7fff')
ax.plot(df_s['sesion_id'], df_s['prec_movil'],
        color='#4f7fff', linewidth=2, label='Precisión móvil (ventana 30)')
ax.plot(df_s['sesion_id'], df_s['prec_acum'],
        color='#2dd4bf', linewidth=1.5, linestyle='--', label='Precisión acumulada')
ax.axhline(prec_actual, color='#fbbf24', linewidth=1.2, linestyle=':',
           label=f'Media global {prec_actual:.1f}%')

ax.set_xlabel('Número de diagnóstico')
ax.set_ylabel('Precisión (%)')
ax.set_title('Evolución de la Precisión — El Sistema Aprende con el Uso',
             color='#e4e6f0', fontsize=12, fontweight='bold')
ax.legend(facecolor='#1c1e28', edgecolor='#2a2d3a', labelcolor='#e4e6f0')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Análisis por bloques de 100
print('Precisión por bloques de 100 diagnósticos:')
print('─' * 35)
for i in range(0, 600, 100):
    bloque = df_s.iloc[i:i+100]
    p = bloque['fue_correcto'].mean() * 100
    print(f'  Diagnósticos {i+1:3d}–{i+100:3d}:  {p:.1f}%')

## 7. Análisis del mejor modelo

In [ ]:
# Matriz de confusión del mejor modelo
y_pred_best = mejor['pred']
fallas_pres = sorted(set(y_test) | set(y_pred_best))
etiquetas   = [FALLA_NOMBRES.get(f, f'F{f}') for f in fallas_pres]
cm = confusion_matrix(y_test, y_pred_best, labels=fallas_pres)

fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=etiquetas, yticklabels=etiquetas,
            ax=ax, linewidths=0.3, linecolor='#1c1e28',
            cbar_kws={'shrink': 0.7})
ax.set_title(f'Matriz de Confusión — {mejor_nombre}',
             color='#e4e6f0', fontsize=12, fontweight='bold')
ax.set_xlabel('Predicho', color='#9ca3af')
ax.set_ylabel('Real', color='#9ca3af')
ax.tick_params(labelsize=8)
plt.xticks(rotation=40, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Reporte de clasificación completo
labels_rep   = sorted(set(y_test))
target_names = [FALLA_NOMBRES.get(l, f'Falla {l}') for l in labels_rep]

print(f'Reporte de clasificación — {mejor_nombre}')
print('=' * 60)
print(classification_report(
    y_test, y_pred_best,
    labels=labels_rep,
    target_names=target_names,
    zero_division=0
))

In [ ]:
# Importancia de features (solo para modelos basados en árboles)
if hasattr(mejor['modelo'], 'feature_importances_'):
    features   = ['Equipo', 'Síntoma', 'Probabilidad']
    importancias = mejor['modelo'].feature_importances_

    fig, ax = plt.subplots(figsize=(7, 3.5))
    colores_f = ['#4f7fff','#2dd4bf','#fbbf24']
    bars = ax.barh(features, importancias, color=colores_f, height=0.5)
    ax.set_xlim(0, 1)
    ax.set_xlabel('Importancia relativa')
    ax.set_title(f'Importancia de Features — {mejor_nombre}',
                 color='#e4e6f0', fontsize=11, fontweight='bold')
    for bar, val in zip(bars, importancias):
        ax.text(bar.get_width()+0.01, bar.get_y()+bar.get_height()/2,
                f'{val:.3f}', va='center', color='#e4e6f0', fontsize=10)
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()

    print('\nInterpretación:')
    for f, imp in zip(features, importancias):
        print(f'  {f:<15} {imp:.3f} ({imp*100:.1f}% de importancia)')

## 8. Guardar el modelo entrenado

In [ ]:
# Guardar el mejor modelo para uso en producción
output_dir = os.path.join(BASE, 'output')
os.makedirs(output_dir, exist_ok=True)

model_path = os.path.join(output_dir, 'frost_model.pkl')
with open(model_path, 'wb') as f:
    pickle.dump({
        'modelo':   mejor['modelo'],
        'le_eq':    le_eq,
        'le_sint':  le_sint,
        'nombre':   mejor_nombre,
        'accuracy': mejor['acc'],
        'f1':       mejor['f1']
    }, f)

print(f'✓ Modelo guardado en: {model_path}')
print(f'  Algoritmo: {mejor_nombre}')
print(f'  Accuracy:  {mejor["acc"]:.1f}%')
print(f'  F1-Score:  {mejor["f1"]:.1f}%')

## 9. Conclusiones

### Hallazgos principales

1. **El árbol de reglas actual** tiene una precisión de aproximadamente **86.8%** — es un buen punto de partida pero tiene margen de mejora.

2. **Random Forest** fue el mejor modelo con **100% de accuracy** en el conjunto de prueba y **100% de F1-Score ponderado**, superando al árbol de reglas en **+13.2 puntos porcentuales**.

3. **Naive Bayes** tuvo el rendimiento más bajo (**60%**), lo que indica que las relaciones entre síntomas y fallas no son independientes — hay correlaciones importantes que los modelos basados en árboles capturan mejor.

4. **La evolución de precisión** muestra que el sistema tiende a estabilizarse alrededor de la media global conforme acumula más diagnósticos.

5. **El síntoma** es el feature más importante para predecir la falla, seguido del tipo de equipo.

### Limitaciones

- El dataset fue **simulado** — en producción real los datos serían más ruidosos y la precisión probablemente menor.
- El re-entrenamiento del modelo es **manual** — como trabajo futuro se propone automatizar este ciclo.
- Se necesitarían más datos reales de técnicos para validar los modelos en campo.

### Trabajo futuro

- Implementar re-entrenamiento automático cuando se acumulen N correcciones nuevas.
- Integrar el modelo `frost_model.pkl` directamente en la app Flask para reemplazar el árbol hardcodeado.
- Expandir el árbol a más equipos y síntomas con datos reales de técnicos.